# NPE tuning stage 3

In [1]:
import numpy as np
from scipy import stats
import torch
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.distributions import Uniform
import sbi
from sbi.utils.user_input_checks import MultipleIndependent
from sbi.neural_nets import posterior_nn
from sbi.neural_nets.embedding_nets import FCEmbedding
from sbi.inference import NPE_C
from sbi.diagnostics import run_sbc, check_sbc
import warnings
import sys
sys.path.append('../../pysimARG')
from discrete_uniform import DiscreteUniform
from LeaveLengthOut_NN import LeaveLengthOut_NN

torch_device = "cpu"

warnings.filterwarnings("ignore", category=UserWarning)

c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load simulation data

Load genome data and clonal tree.

In [2]:
drop_col = range(16, 32)
theta_test = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta_sbc.csv', delimiter=",")
x_test = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x_sbc.csv', delimiter=",")
x_test = np.delete(x_test, drop_col, axis=1)
print(theta_test.shape, x_test.shape)

nan_row_test = np.where(np.isnan(x_test) | np.isinf(x_test))[0]
print(nan_row_test)

theta_test = np.delete(theta_test, nan_row_test, axis=0)
theta_test = torch.tensor(theta_test, device=torch_device)
theta_test = theta_test.to(torch.float32)
theta_test_numpy = theta_test.cpu().numpy()

x_test = np.delete(x_test, nan_row_test, axis=0)
x_test = torch.tensor(x_test, device=torch_device)
x_test = x_test.to(torch.float32)
x_test_numpy = x_test.cpu().numpy()

print(theta_test.shape, x_test.shape)

(1000, 3) (1000, 30)
[865 899]
torch.Size([998, 3]) torch.Size([998, 30])


In [3]:
theta1 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta1.csv', delimiter=",")
x1 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x1.csv', delimiter=",")
theta2 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta2.csv', delimiter=",")
x2 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x2.csv', delimiter=",")

x = np.vstack([x1, x2])
x = np.delete(x, drop_col, axis=1)
theta = np.vstack([theta1, theta2])
print(theta.shape, x.shape)

nan_row = np.where(np.isnan(x) | np.isinf(x))[0]
print(nan_row)

theta = np.delete(theta, nan_row, axis=0)
theta = torch.tensor(theta[:10000, :], device=torch_device)
theta = theta.to(torch.float32)
theta_numpy = theta.cpu().numpy()

x = np.delete(x, nan_row, axis=0)
x = torch.tensor(x[:10000, :], device=torch_device)
x = x.to(torch.float32)
x_numpy = x.cpu().numpy()

print(theta.shape, x.shape)

(20000, 3) (20000, 30)
[  114   681   706  1448  2554  2818  7211  7282  7329  7392  8938  9827
  9973 10223 10788 12192 13567 14388 14653]
torch.Size([10000, 3]) torch.Size([10000, 30])


## Test functions

In [4]:
def SBC_KStest(ranks, num_posterior_samples, parameter_labels):
    num_dimensions = ranks.shape[1] 

    ks_results = []
    p_values = []
    for dim in range(num_dimensions):
        normalized_ranks = ranks[:, dim] / num_posterior_samples
        ks_stat, p_value = stats.kstest(normalized_ranks, 'uniform')
        ks_results.append(ks_stat)
        p_values.append(p_value)
    
    return ks_results, p_values

In [5]:
def mahalanobis_error(theta_est_post, theta_test_numpy):
    maha_errors = np.full((theta_est_post.shape[0]), np.nan)
    for i in range(theta_est_post.shape[0]):
        samples = theta_est_post[i]
        truth = theta_test_numpy[i]
        post_mean = np.mean(samples, axis=0)
        cov_matrix = np.cov(samples, rowvar=False)

        try:
            inv_cov = np.linalg.inv(cov_matrix)
        except np.linalg.LinAlgError:
            print(f"Warning: Singular covariance matrix at index {i}, returning NaN.")
            continue
        
        diff = post_mean - truth
        maha_dist_sq = np.dot(np.dot(diff, inv_cov), diff)
        maha_errors[i] = np.sqrt(maha_dist_sq)
    return maha_errors

## Tuning setting

In [6]:
seeds = [1, 2, 3, 4, 5]
output_dim = [2, 4, 8, 16]
num_posterior_samples=1000
stage3_p_values = np.full((len(seeds), 3, len(output_dim)), np.nan)
stage3_D_stats = np.full((len(seeds), 3, len(output_dim)), np.nan)
stage3_maha_errors = np.full((len(seeds), len(output_dim)), np.nan)
stage3_nll = np.full((len(seeds), len(output_dim)), np.nan)

## Baseline NPE

In [7]:
prior_rho = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_theta = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_L = DiscreteUniform(low=torch.tensor([100.0]), high=torch.tensor([10000.0]))
prior = MultipleIndependent(
    dists=[prior_rho, prior_theta, prior_L],
    validate_args=False,
    device=torch_device
)

In [8]:
for k in range(len(output_dim)):
    num_outputs = output_dim[k]
    embedding_net = LeaveLengthOut_NN(
        input_dim=30,
        num_hiddens=48,
        num_hidden_layers=2,
        num_outputs=num_outputs)
    neural_posterior = posterior_nn(
        model="nsf",
        embedding_net=embedding_net
    )
    print(f"Running output dimension set {num_outputs}...")
    
    for i in range(len(seeds)):
        print(f"Running seed {seeds[i]}...")
        seed = seeds[i]
        torch.manual_seed(seed)
        np.random.seed(seed)

        inference_baseline = NPE_C(prior=prior, density_estimator=neural_posterior, device=torch_device)
        density_estimator_baseline = inference_baseline.append_simulations(theta, x).train(
            max_num_epochs=500
        )
        posterior_baseline = inference_baseline.build_posterior(density_estimator_baseline)

        theta_est_post = np.full((theta_test.shape[0], num_posterior_samples, 3), np.nan)
        for j in tqdm(range(theta_test.shape[0]), desc="Sampling posterior"):
            theta_post = posterior_baseline.sample((num_posterior_samples,), x=x_test[j, :],
                                                show_progress_bars=False, reject_outside_prior=False)
            theta_est_post[j, :, :] = theta_post.detach().numpy()

        parameter_labels = [r"for $\rho_s$", r"for $\theta_s$", r"for L"]
        theta_test_expanded = theta_test.unsqueeze(1)
        theta_est_post_tensor = torch.tensor(theta_est_post, device=torch_device)
        theta_est_post_tensor = theta_est_post_tensor.to(torch.float32)
        is_less_than_truth = theta_est_post_tensor < theta_test_expanded
        ranks = torch.sum(is_less_than_truth, dim=1)

        ks_results, p_values = SBC_KStest(ranks, num_posterior_samples, parameter_labels)
        stage3_p_values[i, :, k] = p_values
        stage3_D_stats[i, :, k] = ks_results

        stage3_maha_errors[i, k] = np.mean(mahalanobis_error(theta_est_post, theta_test_numpy))

        lp = density_estimator_baseline.log_prob(theta_test, x_test)
        stage3_nll[i, k] = -lp.detach().cpu().mean().item()

Running output dimension set 2...
Running seed 1...
 Neural network successfully converged after 96 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:22<00:00, 44.17it/s]


Running seed 2...
 Neural network successfully converged after 44 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:20<00:00, 48.28it/s]


Running seed 3...
 Neural network successfully converged after 78 epochs.

...

Sampling posterior: 100%|██████████| 998/998 [00:20<00:00, 47.67it/s]


Running seed 4...
 Neural network successfully converged after 145 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:20<00:00, 48.83it/s]


Running seed 5...
 Neural network successfully converged after 151 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:20<00:00, 47.87it/s]


In [9]:
np.save('../../data/NPE_tuning/stage3_p_values.npy', stage3_p_values)
np.save('../../data/NPE_tuning/stage3_D_stats.npy', stage3_D_stats)
np.save('../../data/NPE_tuning/stage3_maha_errors.npy', stage3_maha_errors)
np.save('../../data/NPE_tuning/stage3_nll.npy', stage3_nll)

In [10]:
print(np.mean(stage3_nll, axis=0))
print(np.median(stage3_nll, axis=0))

[-3.9141602  -4.50357218 -3.30409317 -3.76350751]
[-3.90006018 -4.59102058 -3.26964116 -3.78364086]


In [11]:
print(np.mean(stage3_maha_errors, axis=0))
print(np.median(stage3_maha_errors, axis=0))

[0.94649828 1.19201549 1.01914906 1.09012324]
[0.96311693 1.19939137 0.9731082  1.10847159]


Conclusion: choose `num_outputs = 2, 4`.